# 92 — Build Geode shot catalog from Excel

Reads the updated Jochen Geode metadata workbook and writes clean Geode metadata tables into
`lbssp_shot_catalog.sqlite`.

This version assumes the Jochen workbook now contains a corrected UTC column on every Geode survey sheet:

```text
geode_final_trigger_utc
```

This column is treated as the authoritative UTC time of the **final trigger in the Geode stack**.

No clock corrections are applied in this notebook.
The raw laptop/file time column `geode_laptop_starttime_iso` is retained for provenance only.

## 1. Configuration

In [1]:
from pathlib import Path
import sqlite3
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("/Volumes/tachyon/LBSSP_DATA")
BOX_ROOT = Path("/Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP")
GLENN_XLSX = BOX_ROOT / "04_FieldData" / "glenn_smartsolo_nodal_metadata_with_estimated_coords.xlsx"
CATALOG_DB = PROJECT_ROOT / "catalog" / "lbssp_shot_catalog.sqlite"

JOCHEN_XLSX = BOX_ROOT / "04_FieldData" / "jochen_field_notes_metadata_tables_with_geode_times.xlsx"

# Fallback for testing if the workbook was uploaded into the current notebook/session.
if not JOCHEN_XLSX.exists():
    JOCHEN_XLSX = Path("/mnt/data/jochen_field_notes_metadata_tables_with_geode_times.xlsx")

OUT_ROOT = PROJECT_ROOT / "geode_catalog_exports"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print("CATALOG_DB:", CATALOG_DB)
print("JOCHEN_XLSX:", JOCHEN_XLSX, JOCHEN_XLSX.exists())
print("OUT_ROOT:", OUT_ROOT)

CATALOG_DB: /Volumes/tachyon/LBSSP_DATA/catalog/lbssp_shot_catalog.sqlite
JOCHEN_XLSX: /Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP/04_FieldData/jochen_field_notes_metadata_tables_with_geode_times.xlsx True
OUT_ROOT: /Volumes/tachyon/LBSSP_DATA/geode_catalog_exports


## 2. Survey definitions

In [ ]:
SURVEY_DEFS = {
    "T1_1m_Refraction": {
        "survey": "T1_1m_refraction",
        "line": "T1",
        "survey_type": "refraction",
        "source_x_col": "source_position_m",
        "n_stack_col": "n_blows",
        "default_source_type": "hammer",
        "seconds_per_blow": 6.0,
        "min_stack_duration_s": 20.0,
        "max_stack_duration_s": 180.0,
        "final_margin_s": 2.0,
        "nodal_timewindow_label": "T1_N2_Refraction1m",
    },
    "T1_2m_Refraction": {
        "survey": "T1_2m_refraction",
        "line": "T1",
        "survey_type": "refraction",
        "source_x_col": "source_position_m",
        "n_stack_col": "n_blows",
        "default_source_type": "hammer",
        "seconds_per_blow": 6.0,
        "min_stack_duration_s": 20.0,
        "max_stack_duration_s": 180.0,
        "final_margin_s": 2.0,
        "nodal_timewindow_label": "T1_N2_Refraction2m",
    },
    "T1_Streamer_MASW": {
        "survey": "T1_streamer_masw",
        "line": "T1",
        "survey_type": "streamer_masw",
        "source_x_col": "shot_location_m",
        "n_stack_col": "n_shots",
        "default_source_type": "PEG",
        "seconds_per_blow": 10.0,
        "min_stack_duration_s": 20.0,
        "max_stack_duration_s": 180.0,
        "final_margin_s": 2.0,
        "nodal_timewindow_label": "T1_N1_Streamer",
    },
    "T2_Streamer_MASW": {
        "survey": "T1A_streamer_masw",
        "line": "T1A",
        "survey_type": "streamer_masw",
        "source_x_col": "shot_location_m",
        "n_stack_col": "n_shots",
        "default_source_type": "PEG",
        "seconds_per_blow": 10.0,
        "min_stack_duration_s": 20.0,
        "max_stack_duration_s": 180.0,
        "final_margin_s": 2.0,
        "nodal_timewindow_label": None,
    },
    "T3_1m_Refraction": {
        "survey": "T3_1m_refraction",
        "line": "T3",
        "survey_type": "refraction",
        "source_x_col": "source_position_m",
        "n_stack_col": "n_blows",
        "default_source_type": "hammer",
        "seconds_per_blow": 6.0,
        "min_stack_duration_s": 20.0,
        "max_stack_duration_s": 180.0,
        "final_margin_s": 2.0,
        "nodal_timewindow_label": "T3_N4_Refraction1am",
    },
    "T4_1m_Refraction": {
        "survey": "T4_1m_refraction",
        "line": "T4",
        "survey_type": "refraction",
        "source_x_col": "source_position_m",
        "n_stack_col": "n_blows",
        "default_source_type": "hammer",
        "seconds_per_blow": 6.0,
        "min_stack_duration_s": 20.0,
        "max_stack_duration_s": 180.0,
        "final_margin_s": 2.0,
        "nodal_timewindow_label": None,
    },
}

## 3. Helper functions

In [3]:
def clean_str(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    return None if s == "" or s.lower() == "nan" else s


def to_num(x):
    return pd.to_numeric(pd.Series([x]), errors="coerce").iloc[0]


def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def parse_utc_datetime(value):
    if pd.isna(value):
        return pd.NaT

    t = pd.to_datetime(value, errors="coerce", utc=False)
    if pd.isna(t):
        return pd.NaT

    t = pd.Timestamp(t)
    if t.tzinfo is None:
        return t.tz_localize("UTC")
    return t.tz_convert("UTC")


def stack_duration_s(n_stack, seconds_per_blow, min_s, max_s):
    n = to_num(n_stack)
    if not np.isfinite(n) or n <= 1:
        return float(min_s)

    return float(min(max_s, max(min_s, (n - 1) * seconds_per_blow)))


def get_source_x(row, df, cfg):
    candidates = [
        cfg.get("source_x_col"),
        "source_x_m",
        "source_position_m",
        "shot_location_m",
        "shot_x_m",
        "source_location_m",
    ]
    for c in candidates:
        if c and c in df.columns:
            val = to_num(row.get(c))
            if np.isfinite(val):
                return float(val), c
    return np.nan, None


def get_n_stack(row, df, cfg):
    candidates = [
        cfg.get("n_stack_col"),
        "n_stack_shots",
        "n_blows",
        "n_shots",
        "shots",
        "blows",
    ]
    for c in candidates:
        if c and c in df.columns:
            val = to_num(row.get(c))
            if np.isfinite(val):
                return int(val), c
    return None, None


def read_receiver_field(row, name):
    val = to_num(row.get(name))
    return float(val) if np.isfinite(val) else None

## 4. Read workbook and build `geode_events`

In [4]:
xl = pd.ExcelFile(JOCHEN_XLSX)
print("Workbook sheets:")
print(xl.sheet_names)

rows = []
sheet_diagnostics = []

for sheet, cfg in SURVEY_DEFS.items():
    if sheet not in xl.sheet_names:
        print(f"WARNING: sheet not found: {sheet}")
        sheet_diagnostics.append({
            "sheet": sheet,
            "status": "missing_sheet",
            "n_rows": 0,
            "n_times": 0,
            "n_source_x": 0,
        })
        continue

    df = pd.read_excel(JOCHEN_XLSX, sheet_name=sheet)
    df = df.dropna(how="all").copy()

    print(f"\nSheet {sheet}: {len(df)} rows")
    print("columns:", list(df.columns))

    if "geode_final_trigger_utc" not in df.columns:
        raise RuntimeError(
            f"{sheet} does not contain required column geode_final_trigger_utc"
        )

    n_times = 0
    n_source = 0

    for idx, r in df.iterrows():
        file_no = to_num(r.get("file_no"))
        shot_no = to_num(r.get("shot_no"))

        source_x, source_x_col_used = get_source_x(r, df, cfg)
        if np.isfinite(source_x):
            n_source += 1

        n_stack, n_stack_col_used = get_n_stack(r, df, cfg)

        final_utc = parse_utc_datetime(r.get("geode_final_trigger_utc"))
        if pd.notna(final_utc):
            n_times += 1

        raw_time_col = first_existing_col(
            df,
            ["geode_laptop_starttime_iso", "geode_laptop_starttime", "shot_time_local"],
        )
        raw_time = r.get(raw_time_col) if raw_time_col else None
        raw_time_parsed = pd.to_datetime(raw_time, errors="coerce") if raw_time is not None else pd.NaT

        dur = stack_duration_s(
            n_stack,
            cfg["seconds_per_blow"],
            cfg["min_stack_duration_s"],
            cfg["max_stack_duration_s"],
        )

        if pd.notna(final_utc):
            stack_end = final_utc + pd.to_timedelta(cfg["final_margin_s"], unit="s")
            stack_start = final_utc - pd.to_timedelta(dur, unit="s")
        else:
            stack_start = pd.NaT
            stack_end = pd.NaT

        src_type = clean_str(r.get("source_type")) or cfg["default_source_type"]

        if np.isfinite(file_no):
            event_id = f"GEODE_{cfg['survey'].upper()}_F{int(file_no):04d}"
        elif np.isfinite(shot_no):
            event_id = f"GEODE_{cfg['survey'].upper()}_SHOT{int(shot_no):04d}"
        else:
            event_id = f"GEODE_{cfg['survey'].upper()}_ROW{idx+2:04d}"

        rows.append({
            "geode_event_id": event_id,
            "instrument_system": "geode",
            "source_sheet": sheet,
            "line": cfg["line"],
            "survey": cfg["survey"],
            "survey_type": cfg["survey_type"],
            "nodal_timewindow_label": cfg["nodal_timewindow_label"],

            "shot_no": int(shot_no) if np.isfinite(shot_no) else None,
            "file_no": int(file_no) if np.isfinite(file_no) else None,

            "source_x_m": float(source_x) if np.isfinite(source_x) else None,
            "source_x_col_used": source_x_col_used,

            "source_type": src_type,
            "n_stack_shots": int(n_stack) if n_stack is not None else None,
            "n_stack_col_used": n_stack_col_used,
            "n_blows": int(n_stack) if n_stack_col_used == "n_blows" and n_stack is not None else None,
            "n_shots": int(n_stack) if n_stack_col_used == "n_shots" and n_stack is not None else None,

            "operator": clean_str(r.get("operator")),
            "plate_type": clean_str(r.get("plate_type")),
            "comment": clean_str(r.get("comment")) or clean_str(r.get("comments")),
            "source_page": clean_str(r.get("source_page")),
            "confidence": clean_str(r.get("confidence")),
            "review_status": clean_str(r.get("review_status")),

            "geode_laptop_starttime_iso": clean_str(r.get("geode_laptop_starttime_iso")),
            "geode_laptop_time_raw": raw_time_parsed.isoformat() if pd.notna(raw_time_parsed) else None,
            "geode_final_trigger_utc_source": "geode_final_trigger_utc",
            "geode_final_trigger_time_utc": final_utc.isoformat() if pd.notna(final_utc) else None,
            "geode_file_time_meaning": "final_trigger",
            "estimated_stack_duration_s": dur,
            "stack_start_utc": stack_start.isoformat() if pd.notna(stack_start) else None,
            "stack_end_utc": stack_end.isoformat() if pd.notna(stack_end) else None,

            "receiver_first_m": read_receiver_field(r, "receiver_first_m"),
            "receiver_last_m": read_receiver_field(r, "receiver_last_m"),
            "receiver_spacing_m": read_receiver_field(r, "receiver_spacing_m"),
            "nominal_shot_spacing_m": read_receiver_field(r, "nominal_shot_spacing_m"),

            "geode_file_path": clean_str(r.get("geode_file_path")),
            "geode_folder": clean_str(r.get("geode_folder")),
            "geode_read_ok": bool(r.get("geode_read_ok")) if "geode_read_ok" in df.columns and not pd.isna(r.get("geode_read_ok")) else None,
            "geode_read_format": clean_str(r.get("geode_read_format")),
            "geode_n_traces": int(to_num(r.get("geode_n_traces"))) if np.isfinite(to_num(r.get("geode_n_traces"))) else None,
            "geode_sampling_rate_hz": float(to_num(r.get("geode_sampling_rate_hz"))) if np.isfinite(to_num(r.get("geode_sampling_rate_hz"))) else None,
            "geode_duration_s_first_trace": float(to_num(r.get("geode_duration_s_first_trace"))) if np.isfinite(to_num(r.get("geode_duration_s_first_trace"))) else None,
            "geode_match_status": clean_str(r.get("geode_match_status")),
            "geode_match_note": clean_str(r.get("geode_match_note")),

            "extra_json": json.dumps({k: (None if pd.isna(v) else str(v)) for k, v in r.to_dict().items()}, default=str),
        })

    sheet_diagnostics.append({
        "sheet": sheet,
        "status": "ok",
        "n_rows": len(df),
        "n_times": n_times,
        "n_source_x": n_source,
    })

geode_events = pd.DataFrame(rows)
sheet_diagnostics = pd.DataFrame(sheet_diagnostics)

print("\ngeode_events:", len(geode_events))
display(sheet_diagnostics)
display(geode_events.groupby(["survey", "line"]).size().reset_index(name="n"))
display(geode_events.head())

Workbook sheets:
['README', 'Transect_Crosswalk', 'Acquisition_Summary', 'T1_1m_Refraction', 'T1_2m_Refraction', 'T1_Streamer_MASW', 'T2_Streamer_MASW', 'T3_1m_Refraction', 'T4_1m_Refraction', 'Review_Issues', 'Geode_Time_Extraction_Summary']

Sheet T1_1m_Refraction: 46 rows
columns: ['shot_no', 'file_no', 'transect', 'survey', 'source_position_m', 'receiver_first_m', 'receiver_last_m', 'receiver_spacing_m', 'nominal_shot_spacing_m', 'n_blows', 'operator', 'plate_type', 'comment', 'source_page', 'confidence', 'review_status', 'geode_laptop_starttime', 'geode_laptop_starttime_iso', 'geode_final_trigger_utc', 'geode_folder_date', 'geode_folder', 'geode_file_path', 'geode_read_ok', 'geode_read_format', 'geode_n_traces', 'geode_sampling_rate_hz', 'geode_duration_s_first_trace']

Sheet T1_2m_Refraction: 44 rows
columns: ['shot_no', 'file_no', 'transect', 'survey', 'source_position_m', 'receiver_first_m', 'receiver_last_m', 'receiver_spacing_m', 'nominal_shot_spacing_m', 'n_blows', 'operator

,sheet,status,n_rows,n_times,n_source_x
0,T1_1m_Refraction,ok,46,42,46
1,T1_2m_Refraction,ok,44,42,42
2,T1_Streamer_MASW,ok,83,83,83
3,T1A_Streamer_MASW,missing_sheet,0,0,0
4,T3_1m_Refraction,ok,39,39,39
5,T4_1m_Refraction,ok,30,30,30


,survey,line,n
0,T1_1m_refraction,T1,46
1,T1_2m_refraction,T1,44
2,T1_streamer_masw,T1,83
3,T3_1m_refraction,T3,39
4,T4_1m_refraction,T4,30


,geode_event_id,instrument_system,source_sheet,line,survey,survey_type,nodal_timewindow_label,shot_no,file_no,source_x_m,...,geode_file_path,geode_folder,geode_read_ok,geode_read_format,geode_n_traces,geode_sampling_rate_hz,geode_duration_s_first_trace,geode_match_status,geode_match_note,extra_json
0,GEODE_T1_1M_REFRACTION_F3001,geode,T1_1m_Refraction,T1,T1_1m_refraction,refraction,T1_N2_Refraction1m,1.0,3001.0,82.5,...,None,None,None,None,NaN,NaN,NaN,None,None,"{""shot_no"": ""1"", ""file_no"": ""3001"", ""transect""..."
1,GEODE_T1_1M_REFRACTION_F3002,geode,T1_1m_Refraction,T1,T1_1m_refraction,refraction,T1_N2_Refraction1m,2.0,3002.0,82.5,...,None,None,None,None,NaN,NaN,NaN,None,None,"{""shot_no"": ""2"", ""file_no"": ""3002"", ""transect""..."
2,GEODE_T1_1M_REFRACTION_F3003,geode,T1_1m_Refraction,T1,T1_1m_refraction,refraction,T1_N2_Refraction1m,3.0,3003.0,82.5,...,None,None,None,None,NaN,NaN,NaN,None,None,"{""shot_no"": ""3"", ""file_no"": ""3003"", ""transect""..."
3,GEODE_T1_1M_REFRACTION_F3004,geode,T1_1m_Refraction,T1,T1_1m_refraction,refraction,T1_N2_Refraction1m,4.0,3004.0,82.5,...,None,None,None,None,NaN,NaN,NaN,None,None,"{""shot_no"": ""4"", ""file_no"": ""3004"", ""transect""..."
4,GEODE_T1_1M_REFRACTION_F3005,geode,T1_1m_Refraction,T1,T1_1m_refraction,refraction,T1_N2_Refraction1m,5.0,3005.0,82.5,...,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,LBSSP_051826,True,SEG2,72.0,8000.0,0.399875,None,None,"{""shot_no"": ""5"", ""file_no"": ""3005"", ""transect""..."


## 5. Build `geode_time_models`

In [5]:
time_model_rows = []
for sheet, cfg in SURVEY_DEFS.items():
    time_model_rows.append({
        "source_sheet": sheet,
        "survey": cfg["survey"],
        "line": cfg["line"],
        "survey_type": cfg["survey_type"],
        "nodal_timewindow_label": cfg["nodal_timewindow_label"],
        "file_time_meaning": "final_trigger",
        "time_source": "geode_final_trigger_utc",
        "clock_correction_applied_in_notebook_s": 0.0,
        "seconds_per_blow": float(cfg["seconds_per_blow"]),
        "min_stack_duration_s": float(cfg["min_stack_duration_s"]),
        "max_stack_duration_s": float(cfg["max_stack_duration_s"]),
        "final_margin_s": float(cfg["final_margin_s"]),
    })

geode_time_models = pd.DataFrame(time_model_rows)
display(geode_time_models)

,source_sheet,survey,line,survey_type,nodal_timewindow_label,file_time_meaning,time_source,clock_correction_applied_in_notebook_s,seconds_per_blow,min_stack_duration_s,max_stack_duration_s,final_margin_s
0,T1_1m_Refraction,T1_1m_refraction,T1,refraction,T1_N2_Refraction1m,final_trigger,geode_final_trigger_utc,0.0,6.0,20.0,180.0,2.0
1,T1_2m_Refraction,T1_2m_refraction,T1,refraction,T1_N2_Refraction2m,final_trigger,geode_final_trigger_utc,0.0,6.0,20.0,180.0,2.0
2,T1_Streamer_MASW,T1_streamer_masw,T1,streamer_masw,T1_N1_Streamer,final_trigger,geode_final_trigger_utc,0.0,10.0,20.0,180.0,2.0
3,T1A_Streamer_MASW,T1A_streamer_masw,T1A,streamer_masw,None,final_trigger,geode_final_trigger_utc,0.0,10.0,20.0,180.0,2.0
4,T3_1m_Refraction,T3_1m_refraction,T3,refraction,T3_N4_Refraction1am,final_trigger,geode_final_trigger_utc,0.0,6.0,20.0,180.0,2.0
5,T4_1m_Refraction,T4_1m_refraction,T4,refraction,None,final_trigger,geode_final_trigger_utc,0.0,6.0,20.0,180.0,2.0


## 6. QC: required fields

In [6]:
qc = (
    geode_events
    .groupby("survey", dropna=False)
    .agg(
        n=("geode_event_id", "count"),
        n_times=("geode_final_trigger_time_utc", lambda x: pd.Series(x).notna().sum()),
        n_stack_start=("stack_start_utc", lambda x: pd.Series(x).notna().sum()),
        n_source_x=("source_x_m", lambda x: pd.Series(x).notna().sum()),
        n_file_no=("file_no", lambda x: pd.Series(x).notna().sum()),
        min_source_x=("source_x_m", "min"),
        max_source_x=("source_x_m", "max"),
    )
    .reset_index()
)

display(qc)

missing_critical = geode_events[
    geode_events["geode_final_trigger_time_utc"].isna()
    | geode_events["source_x_m"].isna()
].copy()

print("Rows missing critical time or source_x:", len(missing_critical))
display(
    missing_critical[
        [
            "geode_event_id",
            "source_sheet",
            "survey",
            "file_no",
            "source_x_m",
            "geode_laptop_starttime_iso",
            "geode_final_trigger_time_utc",
        ]
    ].head(50)
)

,survey,n,n_times,n_stack_start,n_source_x,n_file_no,min_source_x,max_source_x
0,T1_1m_refraction,46,42,42,46,46,82.5,162.5
1,T1_2m_refraction,44,42,42,42,42,43.0,203.0
2,T1_streamer_masw,83,83,83,83,83,87.0,210.0
3,T3_1m_refraction,39,39,39,39,39,-0.5,75.5
4,T4_1m_refraction,30,30,30,30,30,-2.5,71.5


Rows missing critical time or source_x: 6


,geode_event_id,source_sheet,survey,file_no,source_x_m,geode_laptop_starttime_iso,geode_final_trigger_time_utc
0,GEODE_T1_1M_REFRACTION_F3001,T1_1m_Refraction,T1_1m_refraction,3001.0,82.5,None,None
1,GEODE_T1_1M_REFRACTION_F3002,T1_1m_Refraction,T1_1m_refraction,3002.0,82.5,None,None
2,GEODE_T1_1M_REFRACTION_F3003,T1_1m_Refraction,T1_1m_refraction,3003.0,82.5,None,None
3,GEODE_T1_1M_REFRACTION_F3004,T1_1m_Refraction,T1_1m_refraction,3004.0,82.5,None,None
88,GEODE_T1_2M_REFRACTION_ROW0083,T1_2m_Refraction,T1_2m_refraction,NaN,NaN,None,None
89,GEODE_T1_2M_REFRACTION_ROW0084,T1_2m_Refraction,T1_2m_refraction,NaN,NaN,None,None


## 7. Write SQLite tables and CSV exports

In [7]:
conn = sqlite3.connect(CATALOG_DB)

geode_events.to_sql("geode_events", conn, if_exists="replace", index=False)
geode_time_models.to_sql("geode_time_models", conn, if_exists="replace", index=False)
sheet_diagnostics.to_sql("geode_catalog_sheet_diagnostics", conn, if_exists="replace", index=False)

conn.commit()

geode_events.to_csv(OUT_ROOT / "geode_events.csv", index=False)
geode_time_models.to_csv(OUT_ROOT / "geode_time_models.csv", index=False)
sheet_diagnostics.to_csv(OUT_ROOT / "geode_catalog_sheet_diagnostics.csv", index=False)

print("Wrote SQLite tables:")
print("  geode_events")
print("  geode_time_models")
print("  geode_catalog_sheet_diagnostics")
print("CSV exports:", OUT_ROOT)

OperationalError: unable to open database file

## 8. QC plot: corrected Geode final-trigger times

In [ ]:
plot_df = geode_events.copy()
plot_df["final_dt"] = pd.to_datetime(plot_df["geode_final_trigger_time_utc"], errors="coerce", utc=True)
plot_df["source_x_m"] = pd.to_numeric(plot_df["source_x_m"], errors="coerce")

fig, ax = plt.subplots(figsize=(12, 5))
for survey, sub in plot_df.dropna(subset=["final_dt", "source_x_m"]).groupby("survey"):
    ax.scatter(sub["final_dt"], sub["source_x_m"], s=25, label=survey)

ax.set_title("Geode stack final-trigger times from geode_final_trigger_utc")
ax.set_xlabel("UTC final-trigger time")
ax.set_ylabel("Source x (m)")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()